# Tweet Sentiment Classification with TF–IDF

## Project snapshot

| | |
|---|---|
| **Goal** | Classify Sentiment140 tweets as positive or negative and package the full text workflow for reuse. |
| **Data** | 1.6 million labelled tweets from the Kaggle-hosted Sentiment140 training set. |
| **Approach** | Lightweight tweet cleaning, unigram/bigram TF–IDF, and logistic regression in one sklearn `Pipeline`. |
| **Evaluation** | Intended held-out precision, recall, F1, confusion matrix, error review, and coefficient inspection. |
| **Status** | Workflow implemented but not executed in this repository; no measured performance is claimed here. |

The notebook establishes an interpretable text-classification baseline and keeps preprocessing inside the serialized pipeline so raw tweets can be scored consistently.


## 1. Data and setup

Sentiment140 is a classic Twitter sentiment dataset.
- **Source**: Kaggle dataset `kazanova/sentiment140`
- **Size**: 1,600,000 tweets (training CSV)
- **Labels**: `0` = negative, `4` = positive

> Labels are mapped to **binary**: `0 → 0 (negative)`, `4 → 1 (positive)`.


## 2. Modeling approach

For a first strong baseline on short texts like tweets:
- **TF–IDF** provides a sparse numeric representation of text.
- **Logistic Regression** is fast, strong, and easy to interpret (feature weights).

This fast, interpretable combination provides a useful benchmark before considering more complex language models.


### Runtime and reproducibility

Training on all 1.6 million tweets can be memory-intensive. The stratified sampling control supports quicker local iteration; set `SAMPLE_SIZE=None` to use the full dataset. The setup cell locates the repository root automatically, so the notebook can be launched from its project folder.


In [ ]:
# Core libraries and repository paths
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd

for candidate in (Path.cwd(), *Path.cwd().parents):
    if (candidate / "portfolio_utils").is_dir():
        repo_root = candidate
        break
else:
    raise FileNotFoundError("Could not locate the repository root.")

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from portfolio_utils import ARTIFACTS_DIR, clean_tweet

# Visualization
import matplotlib.pyplot as plt

# Sklearn
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, ConfusionMatrixDisplay

# Reproducibility
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)


In [ ]:
# --- Download the dataset (KaggleHub) ---
# KaggleHub handles downloading and caching datasets without needing a Kaggle API key.
import kagglehub

# Download latest version of the dataset
path = kagglehub.dataset_download("kazanova/sentiment140")
print("Path to dataset files:", path)

# Quick check: list files
os.listdir(path)


### Load the training data

In [ ]:
file_path = os.path.join(path, "training.1600000.processed.noemoticon.csv")

# Sentiment140 training CSV has no headers; we assign them.
columns = ["target", "ids", "date", "flag", "user", "text"]

df = pd.read_csv(file_path, encoding="latin-1", names=columns)
df.head()


### Validate labels and data quality

In [ ]:
# Missing values
df.isnull().sum()


In [ ]:
# Distribution of labels before mapping (0 = negative, 4 = positive)
label_counts = df["target"].value_counts().sort_index()
label_counts


In [ ]:
# Plot label distribution
label_counts.plot(kind="bar")
plt.title("Distribution of Sentiment Labels (Original)")
plt.xlabel("Original label")
plt.ylabel("Count")
plt.show()


### Encode the target (0/4 → 0/1)

In [ ]:
# Convert target labels to binary:
# 0 -> 0 (negative)
# 4 -> 1 (positive)
df["target"] = (df["target"] == 4).astype(int)

df["target"].value_counts()


### Create a stratified development sample

In [ ]:
# If you're iterating locally (e.g., on a laptop), sampling can dramatically reduce runtime.
# Set SAMPLE_SIZE=None to train on the full dataset.
SAMPLE_SIZE = 200_000  # try 200k first; bump up if you have time/memory

if SAMPLE_SIZE is not None and len(df) > SAMPLE_SIZE:
    # Stratified sample: keep label balance
    n_per_class = SAMPLE_SIZE // 2
    df_model = (
        df.groupby("target", group_keys=False)
          .apply(lambda x: x.sample(n=min(n_per_class, len(x)), random_state=RANDOM_SEED))
          .sample(frac=1, random_state=RANDOM_SEED)  # shuffle
          .reset_index(drop=True)
    )
else:
    df_model = df.copy()

df_model.shape, df_model["target"].value_counts()


## 3. Text preprocessing

Tweets contain URLs, mentions, hashtags, and punctuation.  
We do **lightweight cleaning** that works well with TF–IDF:

- lowercase
- replace URLs and @mentions with placeholders (`url`, `user`)
- keep hashtag words (remove `#` symbol)
- keep apostrophes (helps with contractions like *don't*)


In [ ]:
# `clean_tweet` lives in portfolio_utils/text.py so the serialized pipeline
# can import the same preprocessing function during inference.
# Preview cleaning on a few examples.
sample_rows = df_model.sample(5, random_state=RANDOM_SEED)[["text", "target"]].copy()
sample_rows["cleaned"] = sample_rows["text"].map(clean_tweet)
sample_rows


## 4. Train/test split

In [ ]:
X = df_model["text"]
y = df_model["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=RANDOM_SEED,
    stratify=y
)

len(X_train), len(X_test)


## 5. Train and evaluate the pipeline

Using a `Pipeline` keeps text cleaning, vectorization, and the model tied together:
- prevents accidental data leakage
- makes it easy to save/reuse


In [ ]:
pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(
        preprocessor=clean_tweet,
        lowercase=False,  # handled by clean_tweet
        stop_words="english",
        ngram_range=(1, 2),
        min_df=5,
        max_df=0.9
    )),
    ("clf", LogisticRegression(
        max_iter=300,
        solver="liblinear",  # solid default for sparse text baselines
        random_state=RANDOM_SEED
    ))
])

pipeline


In [ ]:
pipeline.fit(X_train, y_train)

pred = pipeline.predict(X_test)
print(classification_report(y_test, pred, digits=4))


In [ ]:
ConfusionMatrixDisplay.from_predictions(y_test, pred)
plt.title("Confusion Matrix (Test Set)")
plt.show()


## 6. Diagnose classification errors

Looking at mistakes helps you understand limitations:
- sarcasm/irony
- ambiguous wording
- slang and misspellings
- context missing in short tweets


In [ ]:
# Show a few misclassified examples
results = pd.DataFrame({
    "text": X_test.values,
    "y_true": y_test.values,
    "y_pred": pred
})
mistakes = results[results["y_true"] != results["y_pred"]].sample(10, random_state=RANDOM_SEED)
mistakes


## 7. Interpret predictive n-grams

For linear models, TF–IDF features have weights (coefficients).  
Positive weights push predictions toward **positive sentiment**, negative weights toward **negative sentiment**.


In [ ]:
# Extract feature names and coefficients
tfidf = pipeline.named_steps["tfidf"]
clf = pipeline.named_steps["clf"]

feature_names = np.array(tfidf.get_feature_names_out())
coefs = clf.coef_.ravel()

top_pos_idx = np.argsort(coefs)[-20:][::-1]
top_neg_idx = np.argsort(coefs)[:20]

top_pos = pd.DataFrame({"ngram": feature_names[top_pos_idx], "weight": coefs[top_pos_idx]})
top_neg = pd.DataFrame({"ngram": feature_names[top_neg_idx], "weight": coefs[top_neg_idx]})

top_pos, top_neg


## 8. Save the inference pipeline

The serialized pipeline includes the importable cleaning function, vectorizer, and classifier, so it can accept raw tweet text when loaded from this repository.


In [ ]:
import joblib

model_path = ARTIFACTS_DIR / "sentiment" / "pipeline.joblib"
model_path.parent.mkdir(parents=True, exist_ok=True)
joblib.dump(pipeline, model_path)

model_path


## 9. Results and takeaways

The notebook implements a complete, reusable baseline: raw tweets flow through cleaning, TF–IDF vectorization, and logistic regression, followed by held-out diagnostics and feature-level interpretation. Because the checked-in notebook has no execution outputs, its precision, recall, and F1 remain **unobserved** until the workflow is run.

## Limitations and next steps

- Sentiment140 labels were inferred from emoticons, so label noise and domain age can limit real-world validity.
- A random split may place near-duplicate or temporally related tweets on both sides; duplicate-aware and time-based checks would strengthen evaluation.
- Report class metrics on both the development sample and full dataset before comparing this baseline with a linear SVM or transformer.
- Add robustness slices for tweet length, URLs, mentions, emoji, negation, and contemporary language drift.
